In [1]:
#=================================================================
# Celda 1 — Setup + cargar parquets raw
#=================================================================
import pandas as pd
import numpy as np
from pathlib import Path
import os
# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

RAW_DIR    = Path("output/economico/01-raw")
SILVER_DIR = Path("output/economico/02-silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

parquets = sorted(RAW_DIR.glob("*.parquet"))
assert len(parquets) > 0, f"❌ No hay parquets en {RAW_DIR}"

dfs_raw = {}
for p in parquets:
    dfs_raw[p.stem] = pd.read_parquet(p)
    print(f"✅ {p.stem}: {dfs_raw[p.stem].shape}")

✅ raw_centros_educativos_ccaa: (180, 9)
✅ raw_compraventas_vivienda: (82440, 14)
✅ raw_densidad_poblacion: (3975, 12)
✅ raw_hipotecas_vivienda: (4460, 12)
✅ raw_ipv_precio_vivienda_anual: (2128, 11)


✅ raw_ipv_precio_vivienda_ccaa: (17024, 11)
✅ raw_renta_media_hogar: (576, 12)
✅ raw_tasa_paro_ccaa: (5040, 10)
✅ raw_turismo_pernoctaciones: (131670, 13)


✅ raw_turismo_viajeros: (136910, 14)


In [2]:
#=================================================================
# Celda 2 — Tasa de paro por CCAA × año
# raw_tasa_paro_ccaa: dim_0=tipo_dato, dim_1=sexo, dim_2=ccaa, dim_3=edad
# año extraído de 'fecha' (no existe col 'anyo' en este parquet)
#=================================================================
df_paro = dfs_raw["raw_tasa_paro_ccaa"].copy()

df_paro_clean = (
    df_paro
    .rename(columns={"dim_0": "tipo_dato", "dim_1": "sexo",
                     "dim_2": "CCAA", "dim_3": "grupo_edad"})
    .assign(año=lambda x: pd.to_numeric(x["fecha"].str[:4], errors="coerce").astype("Int64"))
    .query(
        "tipo_dato.str.contains('paro', case=False, na=False) and "
        "sexo == 'Ambos sexos' and "
        "grupo_edad == 'Total' and "
        "CCAA != 'Total Nacional' and "
        "CCAA == CCAA and "
        "secreto == False"
    )
    .dropna(subset=["valor"])
    .groupby(["CCAA", "año"], as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "tasa_paro"})
    .dropna(subset=["CCAA", "año"])
)

print(f"✅ tasa_paro: {df_paro_clean.shape}")
print(f"   Años: {sorted(df_paro_clean['año'].dropna().unique())}")
print(f"   CCAA: {df_paro_clean['CCAA'].nunique()}")
print(df_paro_clean.head())

✅ tasa_paro: (57, 3)
   Años: [np.int64(2021), np.int64(2022), np.int64(2023)]
   CCAA: 19
        CCAA   año  tasa_paro
0  Andalucía  2021    21.6775
1  Andalucía  2022    19.0225
2  Andalucía  2023    18.1575
3     Aragón  2021    10.1600
4     Aragón  2022     9.3975


In [3]:
#=================================================================
# Celda 3 — IPV precio vivienda anual (nacional × año)
# raw_ipv_precio_vivienda_anual: col 'anyo' ya existe como int
# Filtro: índices_y_tasas=='Media anual', general=='General'
#=================================================================
df_ipv = dfs_raw["raw_ipv_precio_vivienda_anual"].copy()

df_ipv_clean = (
    df_ipv
    .rename(columns={
        "índices_y_tasas": "indicador",
        "general,_vivienda_nueva_y_de_segunda_mano": "tipo_vivienda"
    })
    .query(
        "indicador == 'Media anual' and "
        "tipo_vivienda == 'General' and "
        "secreto == False"
    )
    .assign(año=lambda x: pd.to_numeric(x["anyo"], errors="coerce").astype("Int64"))
    .dropna(subset=["valor"])
    .groupby("año", as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "ipv_vivienda"})
)

print(f"✅ ipv_vivienda: {df_ipv_clean.shape}")
print(f"   Años: {sorted(df_ipv_clean['año'].dropna().unique())}")
print(df_ipv_clean.head())

✅ ipv_vivienda: (19, 2)
   Años: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
    año  ipv_vivienda
0  2007     149.35250
1  2008     149.30765
2  2009     141.39660
3  2010     137.98410
4  2011     127.82430


In [4]:
#=================================================================
# Celda 4 — Compraventas de vivienda por CCAA × año
# raw_compraventas_vivienda: col 'comunidades_y_ciudades_autónomas', 'anyo'
# Filtro: estado_de_la_vivienda=='General', tipo_de_dato=='Número'
#=================================================================
df_cv = dfs_raw["raw_compraventas_vivienda"].copy()

df_cv_clean = (
    df_cv
    .rename(columns={"comunidades_y_ciudades_autónomas": "CCAA"})
    .query(
        "estado_de_la_vivienda == 'General' and "
        "tipo_de_dato == 'Número' and "
        "CCAA != 'Total Nacional' and "
        "CCAA == CCAA and "
        "secreto == False"
    )
    .assign(año=lambda x: pd.to_numeric(x["anyo"], errors="coerce").astype("Int64"))
    .dropna(subset=["valor"])
    .groupby(["CCAA", "año"], as_index=False)["valor"]
    .sum()
    .rename(columns={"valor": "compraventas"})
    .dropna(subset=["CCAA", "año"])
)

print(f"✅ compraventas: {df_cv_clean.shape}")
print(f"   Años: {sorted(df_cv_clean['año'].dropna().unique())}")
print(f"   CCAA: {df_cv_clean['CCAA'].nunique()}")
print(df_cv_clean.head())

✅ compraventas: (380, 3)
   Años: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
   CCAA: 19
        CCAA   año  compraventas
0  Andalucía  2007      167803.0
1  Andalucía  2008      120198.0
2  Andalucía  2009       86640.0
3  Andalucía  2010       83195.0
4  Andalucía  2011       72753.0


In [5]:
#=================================================================
# Celda 5 — Merge final CCAA × año
#=================================================================
df_eco = pd.merge(df_paro_clean, df_cv_clean, on=["CCAA", "año"], how="outer")
df_eco = pd.merge(df_eco, df_ipv_clean, on="año", how="left")

df_eco["CCAA"] = df_eco["CCAA"].str.strip()
df_eco["año"]  = df_eco["año"].astype("Int64")
df_eco = df_eco.sort_values(["CCAA", "año"]).reset_index(drop=True)

print(f"📊 Panel económico final: {df_eco.shape}")
print(f"   Años: {sorted(df_eco['año'].dropna().unique())}")
print(f"   CCAA: {df_eco['CCAA'].nunique()}")
print(f"\n   Nulos por columna:\n{df_eco.isnull().sum()}")
print(f"\n{df_eco.head(10).to_string()}")

📊 Panel económico final: (380, 5)
   Años: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
   CCAA: 19

   Nulos por columna:
CCAA              0
año               0
tasa_paro       323
compraventas      0
ipv_vivienda     19
dtype: int64

        CCAA   año  tasa_paro  compraventas  ipv_vivienda
0  Andalucía  2007        NaN      167803.0     149.35250
1  Andalucía  2008        NaN      120198.0     149.30765
2  Andalucía  2009        NaN       86640.0     141.39660
3  Andalucía  2010        NaN       83195.0     137.98410
4  Andalucía  2011        NaN       72753.0     127.82430
5  Andalucía  2012        NaN       62943.0     110.85755
6  Andalucía  2013        NaN       64131.0      98.32850
7  Andalucía  2014     

In [6]:
#=================================================================
# Celda 6 — Export
#=================================================================
ruta_parquet = SILVER_DIR / "economico_ccaa_año.parquet"
ruta_csv     = SILVER_DIR / "economico_ccaa_año.csv"

df_eco.to_parquet(ruta_parquet, index=False)
df_eco.to_csv(ruta_csv, index=False)

print(f"✅ Guardado en: {SILVER_DIR}")
print(f"   Shape final: {df_eco.shape}")
print(f"   Columnas: {list(df_eco.columns)}")

✅ Guardado en: output/economico/02-silver
   Shape final: (380, 5)
   Columnas: ['CCAA', 'año', 'tasa_paro', 'compraventas', 'ipv_vivienda']
